In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog_name = 'ecommerce'

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_orders_items')

display(df_bronze.limit(5))

In [0]:
df_bronze.printSchema()

In [0]:
# drop duplicates

df_silver = df_bronze.dropDuplicates(['order_id','item_seq'])

# convert 'two' to 2 and cast to integer

df_silver = df_silver.withColumn('quantity',
                     when(col('quantity') == 'Two', 2).otherwise(col('quantity')).cast('int'))

# remove any '$' or other symbols from unit_price, keep only numeric

df_silver = df_silver.withColumn('unit_price', regexp_replace('unit_price','[$]','').cast('double'))   

# remove '%' from discount_pct and cast to double

df_silver = df_silver.withColumn('discount_pct',regexp_replace('discount_pct','%','').cast('double'))

# convert coupon code to lower

df_silver = df_silver.withColumn('coupon_code',lower(col('coupon_code')))

# channel 

df_silver = df_silver.withColumn('channel',
                                 when(col('channel')=='web','Website').when(col('channel')=='app','Mobile').otherwise(col('channel')))

In [0]:
# convert date

df_silver = df_silver.withColumn('dt', to_date('dt','yyyy-MM-dd'))

# convert string to timestamp

df_silver = df_silver.withColumn('order_ts', coalesce(to_timestamp('order_ts','yyyy-MM-dd HH:mm:ss'), to_timestamp('order_ts','dd-MM-yyyy HH:mm')))

# string to integer

df_silver = df_silver.withColumn('item_seq', col('item_seq').cast('int'))

# convert tax_amount

df_silver = df_silver.withColumn('tax_amount',
                                 regexp_replace('tax_amount', r'[^0-9.\-]','').cast('double'))


# add processed time

df_silver = df_silver.withColumn('processed_time', current_timestamp())


In [0]:
display(df_silver.limit(5))

In [0]:
df_silver.printSchema()

In [0]:
# create the final table in the silver layer

df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.silver.slv_orders_items')